# Chapter 7 &mdash; The Language of an NFA: Eclose&ndash;Move&ndash;Eclose

**Concept 7 of the Chapter 7 decomposition:** *The Language of an NFA: $\hat{\delta}$ via $Eclosure$–$\delta$–$Eclosure$*

$\hat{\delta}(q,\varepsilon)=Eclosure(q)$; each symbol is Eclose, move, Eclose.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Delta-Hat-Via-Eclosure/Concept-Delta-Hat-Via-Eclosure.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateNFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateNFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


$\hat{\delta}$ for an NFA takes a **set** to a **set**:

* **basis:** $\hat{\delta}(S,\varepsilon) = Eclosure(S)$;
* **step:** $\hat{\delta}(S, aw) = \hat{\delta}\big(Eclosure(\delta(Eclosure(S),a)),\ w\big)$.

Read the step as the three-phase rhythm **Eclose &ndash; move &ndash; Eclose**: settle where free
moves take you, consume one symbol, settle again.

Acceptance: $w \in L(N)$ iff $\hat{\delta}(Q_0, w) \cap F \neq \emptyset$. Note the
basis is *not* $S$ &mdash; forgetting the initial $Eclosure$ is the classic bug.

## 2. Definitions

### A machine where the initial $Eclosure$ matters

In [ ]:
N = md2mc('''NFA
I : '' -> A
A : '' -> F          !! epsilon alone reaches a final state
A : 0 -> A
''')

### $\hat{\delta}$, with the three-phase step

In [ ]:
def dhat(N, S, w):
    cur = Eclosure(N, set(S))                       # basis
    for a in w:
        moved = {t for q in cur for t in step_nfa(N, q, a)}
        cur = Eclosure(N, moved)                    # Eclose - move - Eclose
    return cur

def acc(N, w): return bool(dhat(N, N["Q0"], w) & N["F"])

### and the buggy version that skips the initial $Eclosure$

In [ ]:
def dhat_buggy(N, S, w):
    cur = set(S)
    for a in w:
        cur = Eclosure(N, {t for q in cur for t in step_nfa(N, q, a)})
    return cur

<!-- nav-strip -->

---

&larr;&nbsp;[Ch7&nbsp;6.&nbsp;$Eclosure$: What $\varepsilon$ Edges Do to Simulation](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Eclosure/Concept-Eclosure.ipynb) &nbsp;&middot;&nbsp; [**Chapter 7** index](https://github.com/ganeshutah/Jove/blob/master/Chapter7-NFA/README.md) &nbsp;&middot;&nbsp; [Ch7&nbsp;8.&nbsp;Subset Construction: Converting an NFA to a DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7-NFA/Concept-Subset-Construction/Concept-Subset-Construction.ipynb)&nbsp;&rarr;

---

## 3. Tests

The basis case is $Eclosure$, not the set itself.

In [ ]:
print("dhat(N, Q0, '')   =", sorted(dhat(N, N["Q0"], '')))
print("Q0 itself         =", sorted(N["Q0"]))
assert dhat(N, N["Q0"], '') == Eclosure(N, N["Q0"])
print("\naccepts epsilon?", acc(N, ''), " -- only because of the initial Eclosure")
assert acc(N, '')

Skipping it gets $\varepsilon$ wrong &mdash; the classic bug, made visible.

In [ ]:
buggy = bool(dhat_buggy(N, N["Q0"], '') & N["F"])
print("correct : accepts '' =", acc(N, ''))
print("buggy   : accepts '' =", buggy)
assert acc(N, '') and not buggy

Our $\hat{\delta}$ agrees with `accepts_nfa` everywhere.

In [ ]:
from itertools import product
sig = sorted(N["Sigma"])                       # this machine's alphabet is just {'0'}
strs = [''.join(p) for k in range(10) for p in product(sig, repeat=k)]
assert all(acc(N, s) == accepts_nfa(N, s) for s in strs)
print("alphabet %s; agrees with accepts_nfa on all %d strings up to length 9"
      % (sig, len(strs)))

And Jove's `run_nfa` is the same function under another name.

In [ ]:
for s in ['', '0', '00', '000']:
    print("%-6r dhat=%-16s run_nfa=%s"
          % (s, sorted(dhat(N, N["Q0"], s)), sorted(run_nfa(N, N["Q0"], s))))
assert all(dhat(N, N["Q0"], s) == run_nfa(N, N["Q0"], s)
           for s in ['', '0', '00', '000'])

## 4. Animation

Eclose, move, Eclose &mdash; the rhythm the animation makes visible.

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)

## 5. Exercises


1. What does `accepts_nfa(N, s, chatty=True)` print? Run it.
2. Give a machine where the *final* $Eclosure$ of a step matters but the initial one does not.
3. Write $\hat{\delta}$ recursively rather than as a loop.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter7-NFA/Concept-Delta-Hat-Via-Eclosure')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')